# Week 4 — Baseline Models

Week 3 produced model-ready data: 1,760 training rows, 440 test rows held back,
a target encoded as 22 integers and a scaler fitted on the training rows alone.
Nothing has been *predicted* yet.

This notebook does the smallest possible amount of modelling, on purpose. It
fits a `DummyClassifier` — a model that ignores every feature and guesses from
the label distribution — and measures it with **k-fold cross-validation**. The
output is one number: the accuracy that every model in Weeks 5 to 8 must beat to
have demonstrated that it learned anything at all.

The order of business:

1. **Why a baseline first** — what an accuracy figure means without one.
2. **The naive baselines** — `most_frequent`, `prior`, `stratified`, `uniform`.
3. **Why one split is not enough** — the same model, ten splits, ten answers.
4. **k-fold cross-validation** — how it works and how to read its output.
5. **Why accuracy alone misleads** — even here, and especially elsewhere.
6. **The number to beat** — written down, and what it does and does not prove.

The test set is *not* touched. Everything below is computed on the 1,760
training rows; `data/processed/test.csv` stays sealed until Week 8.

## 0. Setup

Same pattern as Weeks 1-3: put the repository root on `sys.path`, then import
the logic from `src/` rather than writing it inline. This week's new modules are
`src/models/baseline.py` (`get_baseline_model`) and `src/evaluation/metrics.py`
(`evaluate_model`, `cross_validated_accuracy`, `build_cv`), both covered by
`tests/test_baseline.py`.

The data comes from `data/processed/train.csv`, written by Week 3's notebook —
which is exactly why it was written.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data import DEFAULT_RANDOM_STATE, EXPECTED_LABEL_COUNT, FEATURE_COLUMNS, TARGET_COLUMN
from src.evaluation import DEFAULT_CV_FOLDS, build_cv, cross_validated_accuracy, evaluate_model
from src.models import BASELINE_STRATEGIES, get_baseline_model

pd.set_option("display.width", 120)

print("seed", DEFAULT_RANDOM_STATE, "| folds", DEFAULT_CV_FOLDS, "| classes", EXPECTED_LABEL_COUNT)

seed 42 | folds 5 | classes 22


In [2]:
FEATURES = list(FEATURE_COLUMNS)

train = pd.read_csv(REPO_ROOT / "data" / "processed" / "train.csv")
X_train = train[FEATURES]
y_train = train[TARGET_COLUMN]

assert X_train.shape == (1_760, 7)
assert y_train.nunique() == EXPECTED_LABEL_COUNT

print("training rows:", len(train), "| features:", len(FEATURES))
print("rows per crop:", sorted(y_train.value_counts().unique()))

training rows: 1760 | features: 7
rows per crop: [np.int64(80)]


## 1. Why build a baseline before a "real" model

A model's accuracy is not a property of the model. It is a comparison, and
without a second number the comparison has no other side.

Suppose a random forest reports 97% next week. Is that good? The question cannot
be answered from 97% alone:

* On a dataset where 96% of rows carry one label, 97% is barely better than
  answering "the common one" every time, and the model may be worthless.
* On a dataset with 22 equally common labels, where guessing scores 4.5%, 97% is
  an enormous amount of learned structure.

The **baseline** supplies the missing side. It is a model built to be as
unintelligent as possible while still being a legal model: it obeys the
`fit`/`predict` API, it is trained on the same rows, it is scored by the same
metric — and it never looks at a single feature. Its score is therefore the
value of the *labels' distribution* alone, and any excess above it is the value
of the features.

That gives three things:

1. **A floor.** A "real" model that fails to beat the baseline has learned
   nothing from `N`, `P`, `K`, temperature, humidity, pH and rainfall. It is
   broken, mis-wired, or trained on shuffled labels — and the accuracy alone
   would never have told you.
2. **A unit for improvement.** "97%" is a number; "4.5% → 97%" is a result.
3. **A working end-to-end pipeline, cheaply.** Fitting the dummy exercises the
   whole path — load, split, fit, predict, score — before any modelling
   decision can hide a plumbing bug inside a plausible-looking number.

The baseline is also the cheapest sanity check in machine learning: it takes
milliseconds and catches label mix-ups, broken joins and impossible-looking
scores immediately.

## 2. The naive baselines

`sklearn.dummy.DummyClassifier` is scikit-learn's baseline. `fit` looks *only*
at `y`; the feature matrix is accepted and discarded. `get_baseline_model()`
wraps it with this project's defaults so the same object is used in the notebook
and in the tests.

| Strategy | What it predicts |
| --- | --- |
| `most_frequent` | Always the class that appeared most often in training |
| `prior` | The same labels; differs only in `predict_proba` |
| `stratified` | A random class, drawn in proportion to the training frequencies |
| `uniform` | A random class, every class equally likely |

Start with `most_frequent`, the one this project quotes as *the* baseline.

In [3]:
baseline = get_baseline_model("most_frequent")
baseline.fit(X_train, y_train)

predictions = baseline.predict(X_train)
print("distinct predictions:", set(predictions))
print("first ten:", predictions[:10])

distinct predictions: {np.str_('apple')}
first ten: ['apple' 'apple' 'apple' 'apple' 'apple' 'apple' 'apple' 'apple' 'apple'
 'apple']


One crop, for every one of the 1,760 rows. The model has no mechanism for saying
anything else: it recorded the most common training label and returns it
forever. With 22 crops holding 80 training rows each there is no true majority,
so scikit-learn breaks the tie by class order and `apple` wins.

Proof that the features are irrelevant to it — scramble them completely and the
predictions do not move:

In [4]:
scrambled = X_train.sample(frac=1.0, random_state=0).reset_index(drop=True)
print("predictions identical on scrambled features:",
      np.array_equal(baseline.predict(X_train), baseline.predict(scrambled)))

predictions identical on scrambled features: True


In [5]:
result = evaluate_model(baseline, X_train, y_train)
print("accuracy:", round(result["accuracy"], 4), "on", result["n_samples"], "rows")
print("1 / 22   :", round(1 / EXPECTED_LABEL_COUNT, 4))

accuracy: 0.0455 on 1760 rows
1 / 22   : 0.0455


`4.55%`, and `1/22 = 4.5454...%`. That is not a coincidence, it is arithmetic:
the model is right exactly when the true label happens to be the one class it
always predicts, and that class holds 1/22 of the rows.

Note what was *not* done here: this accuracy was measured on the training rows
themselves. For any real model that would be meaningless — a model can memorise
its training data — but a `DummyClassifier` has nothing to memorise with, so the
number is the same either way. Every score from §4 onward is measured on data
the model was not fitted on.

### The per-class view, four weeks early

`evaluate_model` returns a `classification_report` alongside the accuracy,
deliberately. Precision, recall and F1 get their proper treatment in Week 8;
what matters now is simply *seeing* that one number was hiding 22.

In [6]:
print(result["report"][:1200])

              precision    recall  f1-score   support

       apple       0.05      1.00      0.09        80
      banana       0.00      0.00      0.00        80
   blackgram       0.00      0.00      0.00        80
    chickpea       0.00      0.00      0.00        80
     coconut       0.00      0.00      0.00        80
      coffee       0.00      0.00      0.00        80
      cotton       0.00      0.00      0.00        80
      grapes       0.00      0.00      0.00        80
        jute       0.00      0.00      0.00        80
 kidneybeans       0.00      0.00      0.00        80
      lentil       0.00      0.00      0.00        80
       maize       0.00      0.00      0.00        80
       mango       0.00      0.00      0.00        80
   mothbeans       0.00      0.00      0.00        80
    mungbean       0.00      0.00      0.00        80
   muskmelon       0.00      0.00      0.00        80
      orange       0.00      0.00      0.00        80
      papaya       0.00    

Recall is 1.00 for the single predicted crop and 0.00 for the other 21. The
model finds every apple and misses everything else. "4.55% accuracy" summarised
that as one number; the table shows the shape of the failure.

Keep the shape in mind rather than the values: a *real* model can fail in the
same shape while scoring far higher, and only the per-class view reveals it.

## 3. Why a single train/test split is not enough

Before cross-validation, it is worth seeing the problem it solves.

Split the 1,760 training rows into a smaller training part and a validation
part, fit a `stratified` baseline, and score it. Then do it again with a
different shuffle. Nothing about the model or the data changed — only which rows
landed where.

In [7]:
single_split_scores = []
for seed in range(10):
    inner_train, inner_valid = train_test_split(
        train, test_size=0.2, random_state=seed, stratify=train[TARGET_COLUMN]
    )
    model = get_baseline_model("stratified", random_state=seed)
    model.fit(inner_train[FEATURES], inner_train[TARGET_COLUMN])
    score = evaluate_model(model, inner_valid[FEATURES], inner_valid[TARGET_COLUMN])["accuracy"]
    single_split_scores.append(score)

spread = pd.Series(single_split_scores, index=[f"seed {s}" for s in range(10)])
print(spread.round(4).to_string())
print("\nmin", round(spread.min(), 4), "| max", round(spread.max(), 4),
      "| range", round(spread.max() - spread.min(), 4))

seed 0    0.0653
seed 1    0.0426
seed 2    0.0369
seed 3    0.0341
seed 4    0.0682
seed 5    0.0625
seed 6    0.0568
seed 7    0.0511
seed 8    0.0398
seed 9    0.0227

min 0.0227 | max 0.0682 | range 0.0455


The same model, the same data, ten legitimate splits — and a spread of several
percentage points. Any one of those numbers could have been reported as "the"
accuracy.

For a baseline at 4.5% the spread is small in absolute terms, but the lesson
scales: in Week 6 two candidate models will differ by a fraction of a percent,
and a difference smaller than this kind of split-to-split noise is not a
difference at all. A single split gives one draw from a distribution and no
indication of how wide that distribution is.

Two ways to be misled by one split:

* **Optimism or pessimism.** The validation rows might be unusually easy or
  unusually hard.
* **False comparisons.** Model A beats model B on this split and loses on the
  next, and the choice between them is then made by the shuffle.

## 4. k-fold cross-validation

k-fold cross-validation replaces one split with `k` of them, arranged so that
every row is used for validation exactly once.

With `k = 5`:

```
fold 1:  [ VALID ][ train ][ train ][ train ][ train ]
fold 2:  [ train ][ VALID ][ train ][ train ][ train ]
fold 3:  [ train ][ train ][ VALID ][ train ][ train ]
fold 4:  [ train ][ train ][ train ][ VALID ][ train ]
fold 5:  [ train ][ train ][ train ][ train ][ VALID ]
```

The data is cut into five equal parts. The model is fitted five times — each
time on four parts and scored on the part left out — and five scores come back.
Two properties follow:

* **Every row is predicted once**, by a model that did not see it. Nothing is
  wasted, and no row contributes to both fitting and scoring in the same fold.
* **The result is a distribution, not a point.** Its mean is a better estimate
  than any single split's score, and its standard deviation says how much
  trusting a single split would have cost.

The folds are **stratified** — drawn within each class — for the same reason the
Week 3 split was: with 22 classes an unstratified fold can miss a crop entirely.
`build_cv()` returns the project's splitter, `StratifiedKFold(n_splits=5,
shuffle=True, random_state=42)`.

In [8]:
cv = build_cv()
print(cv)

for fold, (fit_index, valid_index) in enumerate(cv.split(X_train, y_train), start=1):
    counts = y_train.iloc[valid_index].value_counts()
    print(f"fold {fold}: fit on {len(fit_index):>4} rows, validate on {len(valid_index):>3}"
          f" | crops present {counts.size} | rows per crop {counts.min()}-{counts.max()}")

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
fold 1: fit on 1408 rows, validate on 352 | crops present 22 | rows per crop 16-16
fold 2: fit on 1408 rows, validate on 352 | crops present 22 | rows per crop 16-16
fold 3: fit on 1408 rows, validate on 352 | crops present 22 | rows per crop 16-16
fold 4: fit on 1408 rows, validate on 352 | crops present 22 | rows per crop 16-16
fold 5: fit on 1408 rows, validate on 352 | crops present 22 | rows per crop 16-16


Each fold validates on 352 rows — 16 per crop, all 22 crops present — and fits
on the remaining 1,408. `cross_val_score` runs that loop and returns the five
scores; `cross_validated_accuracy()` wraps it and adds the mean and standard
deviation.

**How to read the output.** `cross_val_score` returns a plain NumPy array with
one entry per fold, in fold order. It is not a single number, and reporting only
its mean throws away the more interesting half of the result:

* the **mean** is the estimate of how the model performs on unseen data;
* the **standard deviation** is how much that estimate wobbles depending on
  which rows it was measured on;
* a **single low fold** among four high ones points at a subgroup the model
  handles badly, and is worth investigating rather than averaging away.

Note also that `cross_val_score` *clones* the estimator before every fold, so it
fits a fresh copy each time and the object passed in stays unfitted.

In [9]:
rows = []
for strategy in BASELINE_STRATEGIES:
    outcome = cross_validated_accuracy(get_baseline_model(strategy), X_train, y_train)
    rows.append(
        {
            "strategy": strategy,
            **{f"fold {i}": s for i, s in enumerate(outcome["scores"].round(4), start=1)},
            "mean": round(outcome["mean"], 4),
            "std": round(outcome["std"], 4),
        }
    )

baseline_scores = pd.DataFrame(rows).set_index("strategy")
baseline_scores

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
strategy,,,,,,,
most_frequent,0.0455,0.0455,0.0455,0.0455,0.0455,0.0455,0.0000
stratified,0.0398,0.0540,0.0540,0.0398,0.0483,0.0472,0.0064
uniform,0.0540,0.0341,0.0540,0.0455,0.0455,0.0466,0.0073
prior,0.0455,0.0455,0.0455,0.0455,0.0455,0.0455,0.0000


Read the table row by row:

* **`most_frequent` and `prior`** score 0.0455 in every fold, with a standard
  deviation of exactly zero. On perfectly balanced data, always predicting one
  class is right on exactly 1/22 of any stratified fold — there is nothing left
  to vary.
* **`stratified` and `uniform`** hover around the same value but wobble from
  fold to fold, because they guess randomly. Sometimes they edge above
  `most_frequent`; that is luck, not skill, and it is precisely why the
  deterministic strategy is the one quoted as the baseline.

All four sit at roughly 1/22. With 22 balanced classes that is the ceiling on
what ignorance is worth, and no choice of naive strategy escapes it.

In [10]:
best_strategy = baseline_scores["mean"].idxmax()
BASELINE_ACCURACY = baseline_scores.loc["most_frequent", "mean"]

print("quoted baseline (most_frequent):", BASELINE_ACCURACY, f"({BASELINE_ACCURACY:.2%})")
print("best naive strategy            :", best_strategy,
      f"({baseline_scores.loc[best_strategy, 'mean']:.2%})")
print("theoretical 1 / 22             :", round(1 / EXPECTED_LABEL_COUNT, 4))

quoted baseline (most_frequent): 0.0455 (4.55%)
best naive strategy            : stratified (4.72%)
theoretical 1 / 22             : 0.0455


## 5. Why accuracy alone can mislead

Accuracy is the share of rows predicted correctly. It is the right *first*
metric here — the 22 crops are exactly balanced, so no class can dominate the
average, and every mistake costs the same — but "the right first metric" is not
"the only metric", and three failure modes survive a balanced dataset.

**It hides which classes fail.** §2's report already showed this: 4.55%
accuracy was 100% recall on one crop and 0% on 21. A model at 95% could be
perfect on 21 crops and hopeless on the 22nd, and a farmer growing the 22nd
would not care about the average.

**It assumes every error costs the same.** Recommending `maize` where `rice`
would have thrived wastes a season; recommending a crop that cannot grow in that
soil at all may waste more. Accuracy counts both as one mistake.

**It collapses entirely under imbalance.** This dataset is balanced, so the
point has to be made on a modified version of it — which is worth doing, because
the balance here is a property of how the dataset was *constructed*, not of
farming.

In [11]:
# Reframe the same data as a two-class problem: "is this field suited to rice?"
# 80 of 1,760 training rows say yes, so the classes are 95.5% / 4.5%.
binary_y = np.where(y_train == "rice", "rice", "not rice")
print(pd.Series(binary_y).value_counts().to_string())

imbalanced_baseline = get_baseline_model("most_frequent").fit(X_train, binary_y)
imbalanced = evaluate_model(imbalanced_baseline, X_train, binary_y)

print("\naccuracy:", round(imbalanced["accuracy"], 4))
print(imbalanced["report"])

not rice    1680
rice          80

accuracy: 0.9545
              precision    recall  f1-score   support

    not rice       0.95      1.00      0.98      1680
        rice       0.00      0.00      0.00        80

    accuracy                           0.95      1760
   macro avg       0.48      0.50      0.49      1760
weighted avg       0.91      0.95      0.93      1760



**95.45% accuracy, from a model that has never once predicted `rice`.**

Reported alone, that number would look like a strong result. The report shows
what it really is: recall 1.00 on `not rice`, recall 0.00 on `rice`. The model is
useless for the only question that was asked, and the accuracy is high purely
because the majority class is large.

This is exactly the situation the accuracy metric cannot describe, and it is why
Week 8 introduces **precision** (of the fields I called rice, how many were
rice?) and **recall** (of the fields that were rice, how many did I find?), and
the confusion matrix behind both. The mechanism is worth carrying forward now,
before any real model's high score makes it tempting to stop asking.

## 6. Conclusion — the number every future model must beat

The baseline for this project is:

> ### **4.55%** — 5-fold cross-validated accuracy of a `most_frequent`
> `DummyClassifier` on the 1,760 training rows (`1/22 = 0.0455`, standard
> deviation 0.0000 across folds).

What that number licenses:

* **Any model scoring at or below 4.55% is broken or trivial.** Not
  "underperforming" — broken. It has extracted nothing from seven features that
  Week 2 showed separate the crops almost perfectly. Look for shuffled labels, a
  target accidentally left in the feature matrix and then dropped, a model
  fitted on the wrong array, or a metric computed against the wrong vector.
* **Improvement is measured from here.** Week 5's first real classifier is not
  "97% accurate", it is "97% against a 4.55% floor".

And, stated plainly so the next four weeks are not misread:

**This dataset is famously easy.** The 22 crops are near-perfectly separable on
these seven features — Week 2's class-separation scores showed the features
splitting the classes cleanly — so essentially every real algorithm will land
somewhere around 98-99%+, and the gaps between them will be fractions of a
percent. That is a property of the data, not a sign that the comparisons in
Weeks 5-8 are pointless.

The value in those weeks is in **how** models are compared, tuned and explained:

* choosing an evaluation protocol whose differences are believable (Week 6);
* recognising when a gap is smaller than the fold-to-fold noise measured above,
  and so is not a gap;
* tuning hyperparameters without leaking the test set into the choice (Week 6);
* explaining *why* a model predicts what it predicts (Week 7);
* looking past accuracy to per-class behaviour and the confusion matrix
  (Week 8).

Those skills transfer to datasets where the ceiling is 72%, not 99%. Chasing the
last 0.3% here does not.

In [12]:
print(f"BASELINE TO BEAT: {BASELINE_ACCURACY:.4f}  ({BASELINE_ACCURACY:.2%})")
print("strategy        : most_frequent DummyClassifier")
print(f"protocol        : {DEFAULT_CV_FOLDS}-fold stratified CV, seed {DEFAULT_RANDOM_STATE},"
      " on data/processed/train.csv")

# Guard rails: these must hold for the conclusion above to be true.
assert abs(BASELINE_ACCURACY - 1 / EXPECTED_LABEL_COUNT) < 0.005
assert baseline_scores["mean"].max() < 0.10   # no naive strategy escapes 1/k
assert len(train) == 1_760                    # the test set was never touched

BASELINE TO BEAT: 0.0455  (4.55%)
strategy        : most_frequent DummyClassifier
protocol        : 5-fold stratified CV, seed 42, on data/processed/train.csv


## 7. What this week produced, and what it deliberately did not

**Produced**

* `get_baseline_model(strategy)` — the project's `DummyClassifier` factory.
* `evaluate_model(model, X, y)` — accuracy plus a per-class report, from one
  call, so the two are never separated.
* `cross_validated_accuracy(model, X, y)` and `build_cv()` — 5-fold stratified
  cross-validation with a fixed seed.
* A cross-validated baseline of **4.55%**, and the reasoning that makes it the
  floor for Weeks 5-8.

**Not produced, on purpose**

* No real classifier. Logistic regression, KNN, decision trees and random
  forests are **Week 5**.
* No hyperparameter tuning, no grid search — **Week 6**.
* No feature importance — **Week 7**.
* No precision/recall analysis or confusion matrix beyond the glimpse above —
  **Week 8**.
* No test-set score. `data/processed/test.csv` remains unopened.